# Bounding boxes: pujar document → descarregar JSON

1. Selecciona una GPU a Colab si en tens disponible; també funciona en CPU.
2. Puja `bbox-final.zip` a la primera cel·la.
3. Instal·la les dependències i puja el TIFF/PNG/JPEG.
4. Descarrega el JSON de llegenda i dues columnes.

El ZIP inclou tots dos pesos. No cal Drive, dataset ni entrenar.


In [ ]:
from google.colab import files
from pathlib import Path
import json, subprocess, sys, zipfile
uploaded = files.upload()  # Selecciona bbox-final.zip.
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Puja un únic ZIP: bbox-final.zip')
DESTINATION = Path('/content/final-workflow')
DESTINATION.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        if not (DESTINATION / member.filename).resolve().is_relative_to(DESTINATION.resolve()):
            raise ValueError('Ruta invàlida al ZIP')
    archive.extractall(DESTINATION)
ROOT = DESTINATION / 'bbox-final'
assert (ROOT / 'models/general.pt').is_file()
assert (ROOT / 'models/legend.pt').is_file()
print('Workflow preparat:', ROOT)


In [ ]:
# Conserva el PyTorch CUDA que proporciona Colab.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)


In [ ]:
uploaded = files.upload()  # Selecciona només el TIFF, PNG o JPEG a processar.
if len(uploaded) != 1:
    raise ValueError('Puja una única imatge')
IMAGE = Path(next(iter(uploaded))).resolve()
OUTPUT = Path('/content') / f'{IMAGE.stem}.bboxes.json'
completed = subprocess.run([
    sys.executable, str(ROOT / 'predict.py'), str(IMAGE),
    '--output', str(OUTPUT), '--device', 'auto',
], check=True, capture_output=True, text=True)
result = json.loads(completed.stdout)
print(json.dumps(result, indent=2))
files.download(str(OUTPUT))


In [ ]:
# Per processar una altra imatge, torna a executar només la cel·la anterior.
